# Study Area Definition for Paper 2: Land Use Change Analysis

**Objective**: Define and export the study area for Paper 2, which analyzes land use change in rural Spanish municipalities experiencing demographic dynamics between 2010–2025.

**Approach**: 
- Extract rural municipalities (Rural-Remote and Rural-Accessible) from the comprehensive spatial dataset generated in Paper 1 (Step 4)
- Include all four behavioral groups to ensure comprehensive coverage of rural demographic trajectories:
  - **Reverters** (structural loss in A, recovery in B): primary focus for land use change detection
  - **Dynamisers** (sustained growth in both periods): comparative group showing sustained counterurbanization
  - **Structural depopulation** (loss in both periods): baseline/control group for continued decline
  - **Loses in B** (growth in A, loss in B): comparative case of demographic relapse following initial recovery

**Outputs**:
1. Flat CSV table with demographic and spatial attributes
2. GeoPackage with 5 layers (all municipalities + 4 behavioral groups)
3. Descriptive statistics of the study area

## 1. Setup and Configuration

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print(f"GeoPandas version: {gpd.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# File paths
BASE_DIR = Path(r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain")

INPUT_GPKG = BASE_DIR / "data" / "spatial" / "processed" / "p4_spatial_hotspots.gpkg"
OUTPUT_CSV = BASE_DIR / "data" / "demography" / "derived" / "paper2" / "s0_study_area.csv"
OUTPUT_GPKG = BASE_DIR / "data" / "spatial" / "processed" / "s0_study_area.gpkg"

# Create output directories if they don't exist
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_GPKG.parent.mkdir(parents=True, exist_ok=True)

print(f"Input GeoPackage: {INPUT_GPKG}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Output GeoPackage: {OUTPUT_GPKG}")

# Verify input file exists
if not INPUT_GPKG.exists():
    raise FileNotFoundError(f"Input GeoPackage not found: {INPUT_GPKG}")
else:
    print(f"\n✓ Input file verified: {INPUT_GPKG.name}")

## 2. Load Spatial Data

Load the `lisa_results_rural` layer from the Paper 1 spatial hotspots GeoPackage. This layer contains all 6,723 rural municipalities (Rural-Remote + Rural-Accessible) with complete demographic, spatial, and LISA variables.

In [ ]:
# Load the comprehensive rural dataset
gdf = gpd.read_file(INPUT_GPKG, layer="lisa_results_rural")

print(f"Loaded {len(gdf):,} rural municipalities")
print(f"\nColumns: {len(gdf.columns)}")
print(f"CRS: {gdf.crs}")
print(f"Geometry type: {gdf.geometry.geom_type.unique()}")

In [ ]:
# Display column names
print("Available columns:")
for i, col in enumerate(gdf.columns, 1):
    print(f"  {i:2d}. {col}")

## 3. Study Area Composition

The study area includes all four behavioral groups identified in Paper 1 to ensure comprehensive coverage of rural demographic dynamics. While the primary analytical focus is on reverters (municipalities experiencing demographic recovery in Period B), the inclusion of all groups enables robust comparative analysis.

**Behavioral groups:**
- **Reverses in B (reverters)**: Primary focus - municipalities that reversed demographic loss in Period B
- **Grows in both (dynamisers)**: Sustained growth trajectory - comparative group for counterurbanization patterns
- **Structural depopulation**: Continued decline - baseline/control group
- **Loses in B**: Demographic relapse - municipalities that gained in Period A but lost in Period B, providing insights into the sustainability of initial recovery processes

In [ ]:
# Check distribution before filtering
print("Behavioral group distribution (before filtering):")
print(gdf['behavioural_group'].value_counts().sort_index())
print(f"\nTotal: {len(gdf):,} municipalities")

In [ ]:
# Keep all 4 behavioral groups for comprehensive analysis
study_area = gdf.copy()
print(f"Study area includes all {len(study_area):,} rural municipalities")

In [ ]:
# Distribution after filtering
print("\nBehavioral group distribution (study area):")
print(study_area['behavioural_group'].value_counts().sort_index())

## 4. Descriptive Statistics

Characterize the study area by behavioral group, typology, and territorial distribution.

In [ ]:
# Distribution by typology
print("Distribution by Goerlich typology:")
print(study_area['tipo_goerlich'].value_counts())
print()

In [ ]:
# Cross-tabulation: behavioral group × typology
crosstab = pd.crosstab(
    study_area['behavioural_group'],
    study_area['tipo_goerlich'],
    margins=True
)

print("Behavioral group × Typology:")
print(crosstab)
print()

In [ ]:
# Distribution by CCAA (top 10)
ccaa_counts = study_area['CCAA_Name'].value_counts().head(10)
print("Top 10 Autonomous Communities by municipality count:")
for ccaa, count in ccaa_counts.items():
    pct = 100 * count / len(study_area)
    print(f"  {ccaa:30s}: {count:4,} ({pct:5.1f}%)")
print()

In [ ]:
# Population statistics by behavioral group
print("Population statistics by behavioral group (Period B, 2025):")
pop_stats = study_area.groupby('behavioural_group')['pop_end_B'].agg([
    ('count', 'count'),
    ('total_pop', 'sum'),
    ('mean_pop', 'mean'),
    ('median_pop', 'median'),
    ('min_pop', 'min'),
    ('max_pop', 'max')
]).round(0)

print(pop_stats.to_string())
print()
print(f"Total study area population (2025): {study_area['pop_end_B'].sum():,.0f}")

In [ ]:
# Mean variation rates by behavioral group
print("Mean annual population variation rates (%):")
var_stats = study_area.groupby('behavioural_group')[[
    'var_anual_media_pct_A', 
    'var_anual_media_pct_B'
]].mean().round(2)

var_stats.columns = ['Period A (2010-2017)', 'Period B (2018-2025)']
print(var_stats.to_string())

In [ ]:
# Diagnostic check
print(f"Total rows in gdf: {len(gdf):,}")
print(f"Non-null behavioral groups: {gdf['behavioural_group'].notna().sum():,}")
print(f"NULL behavioral groups: {gdf['behavioural_group'].isna().sum()}")

print("\nBehavioral group distribution:")
print(gdf['behavioural_group'].value_counts(dropna=False).sort_index())

print("\nTypology distribution:")
print(gdf['tipo_goerlich'].value_counts())

# Check if there are any non-rural municipalities that slipped through
print("\nUnique typologies in dataset:")
print(gdf['tipo_goerlich'].unique())

## 5. Export Flat CSV

Export a flat CSV table with all demographic and spatial attributes (without geometry). This table can be easily joined with other datasets (e.g., SIDAMUN socioeconomic variables, land use metrics) using `Mun_Code` as the key.

In [ ]:
# Prepare dataframe without geometry
df_export = study_area.drop(columns='geometry')

# Ensure Mun_Code is string with leading zeros
df_export['Mun_Code'] = df_export['Mun_Code'].astype(str).str.zfill(5)

# Sort by Mun_Code
df_export = df_export.sort_values('Mun_Code').reset_index(drop=True)

print(f"DataFrame shape: {df_export.shape}")
print(f"Columns: {len(df_export.columns)}")

In [ ]:
# Export to CSV
df_export.to_csv(OUTPUT_CSV, index=False, sep=';', encoding='utf-8-sig')

print(f"\n✓ CSV exported: {OUTPUT_CSV}")
print(f"  Rows: {len(df_export):,}")
print(f"  Columns: {len(df_export.columns)}")
print(f"  Size: {OUTPUT_CSV.stat().st_size / 1024 / 1024:.2f} MB")

## 6. Export GeoPackage

Export a GeoPackage with 5 layers:
1. **study_area_all**: All 6,717 municipalities (complete study area)
2. **reverters**: Municipalities that reversed demographic loss in Period B (2,175)
3. **dynamisers**: Municipalities with sustained growth in both periods (807)
4. **structural_depopulation**: Municipalities with continued population loss (3,402)
5. **loses_in_b**: Municipalities with demographic relapse - growth in Period A followed by loss in Period B (333)

In [ ]:
# Ensure Mun_Code is string in GeoDataFrame
study_area['Mun_Code'] = study_area['Mun_Code'].astype(str).str.zfill(5)

# Sort by Mun_Code
study_area = study_area.sort_values('Mun_Code').reset_index(drop=True)

In [ ]:
# Layer 1: All municipalities
study_area.to_file(OUTPUT_GPKG, layer='study_area_all', driver='GPKG')
print(f"✓ Layer 'study_area_all' exported: {len(study_area):,} municipalities")

In [ ]:
# Layer 2: Reverters
reverters = study_area[study_area['behavioural_group'] == 'Reverses in B'].copy()
reverters.to_file(OUTPUT_GPKG, layer='reverters', driver='GPKG')
print(f"✓ Layer 'reverters' exported: {len(reverters):,} municipalities")

In [ ]:
# Layer 3: Dynamisers
dynamisers = study_area[study_area['behavioural_group'] == 'Grows in both'].copy()
dynamisers.to_file(OUTPUT_GPKG, layer='dynamisers', driver='GPKG')
print(f"✓ Layer 'dynamisers' exported: {len(dynamisers):,} municipalities")

In [ ]:
# Layer 4: Structural depopulation
structural_depop = study_area[study_area['behavioural_group'] == 'Structural depopulation'].copy()
structural_depop.to_file(OUTPUT_GPKG, layer='structural_depopulation', driver='GPKG')
print(f"✓ Layer 'structural_depopulation' exported: {len(structural_depop):,} municipalities")

In [ ]:
# Layer 5: Loses in B
loses_in_b = study_area[study_area['behavioural_group'] == 'Loses in B'].copy()
loses_in_b.to_file(OUTPUT_GPKG, layer='loses_in_b', driver='GPKG')
print(f"✓ Layer 'loses_in_b' exported: {len(loses_in_b):,} municipalities")

In [ ]:
# Verify GeoPackage
import fiona

layers = fiona.listlayers(OUTPUT_GPKG)
print(f"\n✓ GeoPackage exported: {OUTPUT_GPKG}")
print(f"  Layers: {len(layers)}")
for layer in layers:
    gdf_check = gpd.read_file(OUTPUT_GPKG, layer=layer)
    print(f"    - {layer}: {len(gdf_check):,} municipalities")
print(f"  Size: {OUTPUT_GPKG.stat().st_size / 1024 / 1024:.2f} MB")

## 7. Summary

**Study Area Definition Complete**

The study area for Paper 2 (land use change analysis) has been defined and exported. This area comprises **6,723 rural municipalities** in Spain, representing the complete rural demographic landscape across both Rural-Accesible and Rural-Remoto typologies.

**Key characteristics:**
- **Reverters (32.4%, n=2,175)**: Primary focus for detecting land use changes associated with demographic recovery. More prevalent in Rural-Accesible municipalities (34.4%) than Rural-Remoto (29.5%).
- **Structural depopulation (50.6%, n=3,402)**: Baseline/control group for continued decline, representing the majority of rural municipalities. Particularly dominant in Rural-Remoto areas (60.1%).
- **Dynamisers (12.0%, n=807)**: Comparative group showing sustained growth, concentrated in Rural-Accesible municipalities (17.0% vs. 5.1% in Rural-Remoto).
- **Loses in B (5.0%, n=333)**: Municipalities with demographic relapse following initial recovery, providing insights into the sustainability of counterurbanization processes.
- **Unclassified (0.1%, n=6)**: Municipalities with incomplete temporal data due to administrative changes during 2010-2025.

**Territorial distribution:**
- Rural-Accesible: 3,885 municipalities (57.8%)
- Rural-Remoto: 2,838 municipalities (42.2%)
- Total population (2025): 6,224,483 inhabitants

**Outputs:**
1. CSV table: `data/demography/derived/paper2/s0_study_area.csv` (6,723 rows × 43 columns)
2. GeoPackage: `data/spatial/processed/s0_study_area.gpkg` (5 layers + study_area_all)

**Next steps:**
- Visual inspection and cartographic representation in QGIS
- Land use metrics extraction (CORINE Land Cover 2018→2024, Sentinel-2 NDVI)
- Comparative analysis of land use change patterns across behavioral groups, with particular focus on reverters vs. structural depopulation contrast

In [ ]:
print("="*80)
print("STUDY AREA DEFINITION - SUMMARY")
print("="*80)
print(f"Total municipalities: {len(study_area):,}")
print(f"Total population (2025): {study_area['pop_end_B'].sum():,.0f}")

print(f"\nBehavioral groups:")
for group, count in study_area['behavioural_group'].value_counts(dropna=False).sort_index().items():
    pct = 100 * count / len(study_area)
    group_str = str(group) if group is not None else "Unclassified"
    print(f"  {group_str:25s}: {count:5,} ({pct:5.1f}%)")

print(f"\nTypologies:")
for typ, count in study_area['tipo_goerlich'].value_counts().items():
    pct = 100 * count / len(study_area)
    print(f"  {typ:25s}: {count:5,} ({pct:5.1f}%)")

print(f"\nBehavioral groups by typology:")
print("-" * 80)
crosstab = pd.crosstab(
    study_area['behavioural_group'],
    study_area['tipo_goerlich'],
    margins=True,
    dropna=False
)
print(crosstab)

print("\nPercentages within each typology:")
print("-" * 80)
crosstab_pct = pd.crosstab(
    study_area['behavioural_group'],
    study_area['tipo_goerlich'],
    normalize='columns',
    dropna=False
) * 100

for typ in ['Rural - Accesible', 'Rural - Remoto']:
    print(f"\n{typ}:")
    for group in crosstab_pct.index:
        val = crosstab_pct.loc[group, typ]
        count = crosstab.loc[group, typ]
        group_str = str(group) if pd.notna(group) else "Unclassified"
        print(f"  {group_str:25s}: {count:5,} ({val:5.1f}%)")

print("="*80)